In [ ]:
import json
import csv
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path

results_file = Path("./outputs/results.csv")

tasks_setup = {
    "rte":  {"lr": 1e-3, "epochs": 20, "metric": "Accuracy"},
    "mrpc": {"lr": 7e-4, "epochs": 20, "metric": "F1"},
    "cola": {"lr": 7e-4, "epochs": 20, "metric": "MCC"},
    "sst2": {"lr": 4e-4, "epochs": 20, "metric": "Accuracy"},
    "qnli": {"lr": 1e-4, "epochs": 20, "metric": "Accuracy"},
    "mnli": {"lr": 1e-4, "epochs": 20, "metric": "Accuracy"},
    "qqp":  {"lr": 4e-4, "epochs": 20, "metric": "F1"},
    "stsb": {"lr": 1e-4, "epochs": 20, "metric": "Spearman"}
}

In [ ]:
def run_task(task, 
             fine_tune_type="bitfit", 
             model="bert-base-cased", 
             seeds=[1, 2, 3, 4, 5], 
             batch=16, 
             gpu=0):

    config = tasks_setup[task]
    metric = config["metric"]
    results = []

    for seed in seeds:
        output_path = Path("./outputs") / f"{fine_tune_type}_{task}_seed{seed}"
        output_path.mkdir(parents=True, exist_ok=True)

        subprocess.run(["python", "run_glue.py",
                        "--output-path", str(output_path),
                        "--task-name", task,
                        "--model-name", model,
                        "--fine-tune-type", fine_tune_type,
                        "--learning-rate", str(2e-5),
                        "--epochs", str(config["epochs"]),
                        "--batch-size", str(batch),
                        "--gpu-device", str(gpu),
                        "--seed", str(seed)])

        with open(output_path / "eval_results.json") as f:
            acc = json.load(f)["validation"][metric]
        results.append(acc)

    mean = np.mean(results) * 100
    std = np.std(results) * 100

    results_file.parent.mkdir(parents=True, exist_ok=True)
    write_header = not results_file.exists()
    with open(results_file, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["model", "task", "fine_tune_type", "metric", "mean", "std"])
        if write_header:
            writer.writeheader()
        writer.writerow({"model": model, 
                         "task": task, 
                         "fine_tune_type": fine_tune_type,
                         "metric": metric, 
                         "mean": mean, 
                         "std": std})

In [3]:
!pip install torch transformers datasets tokenizers scikit-learn scipy sentencepiece sacremoses tqdm matplotlib seaborn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 51.4 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 15.0 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 58.6 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 94.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 96.2 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 56.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 57.3 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 99.4 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 48.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 100.9 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 91.6 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/28

In [17]:
run_task("rte", fine_tune_type="full_ft")

2026-03-12 17:29:07,428 - run_glue - INFO - ############################################################################################
2026-03-12 17:29:07,428 - run_glue - INFO - ############################################################################################
2026-03-12 17:29:07,428 - run_glue - INFO - ############################################################################################
2026-03-12 17:29:07,428 - run_glue - INFO - 
2026-03-12 17:29:07,428 - run_glue - INFO - Training Details: 
2026-03-12 17:29:07,428 - run_glue - INFO - ----------------------------------------------
2026-03-12 17:29:07,428 - run_glue - INFO - Model Name: bert-base-cased
2026-03-12 17:29:07,428 - run_glue - INFO - Task Name: rte
2026-03-12 17:29:07,428 - run_glue - INFO - Fine Tuning Type: full_ft
2026-03-12 17:29:07,428 - run_glue - INFO - Output Directory: outputs/full_ft_rte_seed1
2026-03-12 17:29:07,428 - run_glue - INFO - Running on GPU #0
2026-03-12 17:29:07,428 - run_glue - IN

2026-03-12 17:29:17,470 - _client - INFO - HTTP Request: GET https://huggingface.co/api/models/bert-base-cased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-12 17:29:17,573 - _client - INFO - HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-cased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-12 17:29:17,681 - _client - INFO - HTTP Request: GET https://huggingface.co/api/models/bert-base-cased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-12 17:29:17,786 - _client - INFO - HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-cased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


Map: 4980 examples [00:00, 9575.64 examples/s]         
Map: 554 examples [00:00, 7510.60 examples/s]        
Map: 6000 examples [00:00, 11001.64 examples/s]        


2026-03-12 17:29:20,427 - _client - INFO - HTTP Request: HEAD https://huggingface.co/bert-base-cased/resolve/main/config.json "HTTP/1.1 200 OK"


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5942.04it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider trai

...

2026-03-12 20:01:09,070 - _client - INFO - HTTP Request: GET https://huggingface.co/api/datasets/glue/tree/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c/ax?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-12 20:01:09,178 - _client - INFO - HTTP Request: GET https://huggingface.co/api/datasets/nyu-mll/glue/tree/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c/ax?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-03-12 20:01:09,278 - _client - INFO - HTTP Request: GET https://huggingface.co/api/datasets/glue/tree/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-03-12 20:01:09,435 - _client - INFO - HTTP Request: GET https://huggingface.co/api/datasets/nyu-mll/glue/tree/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c?recursive=false&expand=false "HTTP/1.1 200 OK"
2026-03-12 20:01:09,545 - _client - INFO - HTTP Request: HEAD https://huggingface.co/datasets/glue/resolve/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c/dataset_infos.json "

Map: 4980 examples [00:00, 9773.40 examples/s]         
Map: 554 examples [00:00, 7291.65 examples/s]        
Map: 6000 examples [00:00, 10891.90 examples/s]        


2026-03-12 20:01:13,216 - _client - INFO - HTTP Request: HEAD https://huggingface.co/bert-base-cased/resolve/main/config.json "HTTP/1.1 200 OK"


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4477.61it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider trai



Trainable Components:
----------------------------------------

bert.embeddings.word_embeddings.weight   --->   torch.Size([28996, 768])
bert.embeddings.position_embeddings.weight   --->   torch.Size([512, 768])
bert.embeddings.token_type_embeddings.weight   --->   torch.Size([2, 768])
bert.embeddings.LayerNorm.weight   --->   torch.Size([768])
bert.embeddings.LayerNorm.bias   --->   torch.Size([768])
bert.encoder.layer.0.attention.self.query.weight   --->   torch.Size([768, 768])
bert.encoder.layer.0.attention.self.query.bias   --->   torch.Size([768])
bert.encoder.layer.0.attention.self.key.weight   --->   torch.Size([768, 768])
bert.encoder.layer.0.attention.self.key.bias   --->   torch.Size([768])
bert.encoder.layer.0.attention.self.value.weight   --->   torch.Size([768, 768])
bert.encoder.layer.0.attention.self.value.bias   --->   torch.Size([768])
bert.encoder.layer.0.attention.output.dense.weight   --->   torch.Size([768, 768])
bert.encoder.layer.0.attention.output.dense.bias 

In [ ]:
df = pd.read_csv(results_file)
for index, row in df.iterrows():
    df.loc[index, "result"] = f"{row["mean"]}±{row["std"]}"
df.pivot(index="task", columns="fine_tune_type", values="result")